<a href="https://colab.research.google.com/github/duartegonzaloariel/UDELAR_Astrometric-calibration-/blob/main/01_ajuste_clasico_por_campana.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 · Ajuste clásico por campaña vs Caldas

Ajuste geométrico propio por campaña (estación + año + mes): centro libre, polinomio radial, rotación, paridad e inclinación. Limpieza de identificaciones erróneas con ese mismo ajuste y comparación contra la predicción de Caldas (`Posicion_Xpred`, `Posicion_Ypred`).

Salida: `df_global_con_ajuste.csv` y `ajuste_por_campaña.csv` en Drive, que usa el notebook 02 (red).

## 0. Configuración y carga

In [ ]:
import os, numpy as np, pandas as pd
from scipy.optimize import least_squares
from IPython.display import display
from google.colab import drive
drive.mount('/content/drive')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)
pd.set_option('display.width', None)

PATH_BASE     = "/content/drive/MyDrive/InvestigacionesPropias/Fireball/estrellas/calibracion/"
ARCHIVO       = "df_global.csv"
UMBRAL_ARCMIN = 60.0   # estrellas a más de esto de MI ajuste = identificación errónea
MAX_ITER_OUT  = 5      # iteraciones de (ajustar -> sacar outliers -> reajustar)

# Campañas que no son la cámara estándar o no tienen datos suficientes.
# Se ajustan y se muestran igual, pero quedan fuera de los resúmenes y del notebook 02.
EXCLUIR = {
    '3_2020-05': 'otra resolución (escala ~22 arcmin/px)',
    '7_2021-08': 'pocos puntos (42) y 55% outliers',
}

Mounted at /content/drive


In [ ]:
df = pd.read_csv(os.path.join(PATH_BASE, ARCHIVO))
df['zenith'] = 90.0 - df['altitude']
df['campaña'] = (df['station'].astype(str) + '_' + df['año'].astype(str)
                 + '-' + df['mes'].astype(int).map('{:02d}'.format))
df['excluida'] = df['campaña'].isin(EXCLUIR)

display(df.head())
display(pd.DataFrame({'registros': [len(df)], 'campañas': [df['campaña'].nunique()],
                      'estaciones': [df['station'].nunique()],
                      'registros_excluidos': [int(df['excluida'].sum())]}))

,station,año,mes,Posicion_Y,Posicion_X,azimuth,altitude,Posicion_Ypred,Posicion_Xpred,zenith,campaña,excluida
0,1,2022,9,541,869,158.494,76.494,556,885,13.506,1_2022-09,False
1,1,2022,9,486,867,184.884,77.475,504,881,12.525,1_2022-09,False
2,1,2022,9,586,710,65.015,79.503,604,741,10.497,1_2022-09,False
3,1,2022,9,734,1038,139.870,49.255,729,1056,40.745,1_2022-09,False
4,1,2022,9,279,1191,205.629,35.018,289,1187,54.982,1_2022-09,False


,registros,campañas,estaciones,registros_excluidos
0,8870,34,18,97


## 1. Modelo geométrico

In [ ]:
def vec(z_deg, a_deg):
    """(cenit, azimut) en grados -> vector unitario."""
    z, a = np.deg2rad(z_deg), np.deg2rad(a_deg)
    return np.stack([np.sin(z)*np.cos(a), np.sin(z)*np.sin(a), np.cos(z)], axis=-1)

def sep_arcmin(v1, v2):
    """Separación angular entre vectores unitarios, en arcmin."""
    return np.rad2deg(np.arccos(np.clip(np.sum(v1*v2, axis=-1), -1, 1))) * 60

def modelo(p, x, y, paridad):
    """p = [x0, y0, a0, p1, p2, p3, eps, E]
    1) r, phi respecto del centro (x0, y0)
    2) z' = p1 r + p2 r^2 + p3 r^3 ;  A' = a0 + paridad*phi   (marco de la cámara)
    3) inclinación: el eje óptico se inclina eps grados hacia el azimut E"""
    x0, y0, a0, p1, p2, p3, eps, E = p
    dx, dy = x - x0, y - y0
    r = np.hypot(dx, dy)
    phi = np.rad2deg(np.arctan2(dy, dx))
    v = vec(p1*r + p2*r**2 + p3*r**3, a0 + paridad*phi)
    e, t = np.deg2rad(E), np.deg2rad(eps)
    Rz = lambda g: np.array([[np.cos(g), -np.sin(g), 0], [np.sin(g), np.cos(g), 0], [0, 0, 1]])
    Ry = np.array([[np.cos(t), 0, np.sin(t)], [0, 1, 0], [-np.sin(t), 0, np.cos(t)]])
    return v @ (Rz(e) @ Ry @ Rz(-e)).T

def ajustar(x, y, z, a):
    """Ajusta el modelo probando las dos paridades; devuelve la mejor."""
    v_obs = vec(z, a)
    cx, cy = np.median(x), np.median(y)
    r0 = np.hypot(x - cx, y - cy)
    p1_0 = np.sum(r0*z) / np.sum(r0**2)
    mejor = None
    for par in (+1, -1):
        phi0 = np.rad2deg(np.arctan2(y - cy, x - cx))
        d = np.deg2rad(a - par*phi0)
        a0_0 = np.rad2deg(np.arctan2(np.sin(d).mean(), np.cos(d).mean()))
        p0 = [cx, cy, a0_0, p1_0, 0.0, 0.0, 0.5, 0.0]
        f = lambda p: (modelo(p, x, y, par) - v_obs).ravel()
        res = least_squares(f, p0, x_scale=[50, 50, 10, p1_0, p1_0/500, p1_0/2.5e5, 1, 30],
                            loss='soft_l1', f_scale=np.deg2rad(0.5), max_nfev=4000)
        err = sep_arcmin(modelo(res.x, x, y, par), v_obs)
        if mejor is None or np.median(err) < np.median(mejor['err']):
            mejor = dict(p=res.x, paridad=par, err=err)
    return mejor

## 2. Ajuste por campaña con limpieza de outliers

Se ajusta, se sacan las estrellas a más de `UMBRAL_ARCMIN` y se vuelve a ajustar, hasta que no cambia.

Error de Caldas: distancia entre (X, Y) medido y (Xpred, Ypred), pasada a arcmin con la geometría local del ajuste.

In [ ]:
filas, params = [], {}
for col in ['err_mio', 'err_caldas', 'err_caldas_px']:
    df[col] = np.nan
df['outlier'] = False

for camp, g in df.groupby('campaña'):
    idx = g.index.values
    x, y = g['Posicion_X'].values.astype(float), g['Posicion_Y'].values.astype(float)
    z, a = g['zenith'].values, g['azimuth'].values
    keep = np.ones(len(g), bool)
    for _ in range(MAX_ITER_OUT):
        fit = ajustar(x[keep], y[keep], z[keep], a[keep])
        err = sep_arcmin(modelo(fit['p'], x, y, fit['paridad']), vec(z, a))
        nuevo = err <= UMBRAL_ARCMIN
        if nuevo.sum() < 10 or np.array_equal(nuevo, keep):
            break
        keep = nuevo
    p, par = fit['p'], fit['paridad']
    params[camp] = dict(p=p, paridad=par)

    xp, yp = g['Posicion_Xpred'].values.astype(float), g['Posicion_Ypred'].values.astype(float)
    err_cal = sep_arcmin(modelo(p, x, y, par), modelo(p, xp, yp, par))
    df.loc[idx, 'err_mio'] = err
    df.loc[idx, 'err_caldas'] = err_cal
    df.loc[idx, 'err_caldas_px'] = np.hypot(x - xp, y - yp)
    df.loc[idx, 'outlier'] = ~keep

    filas.append(dict(
        campaña=camp, excluida=camp in EXCLUIR, N=len(g), N_ok=int(keep.sum()),
        pct_out=100*(1 - keep.mean()), paridad=par, escala=p[3]*60,
        x0=p[0], y0=p[1], a0=p[2] % 360, incl=abs(p[6]), E=p[7] % 360,
        p1=p[3], p2=p[4], p3=p[5],
        mio_med=np.median(err[keep]), mio_mae=np.mean(err[keep]),
        cal_med=np.median(err_cal[keep]), cal_mae=np.mean(err_cal[keep]),
        cal_med_px=np.median(np.hypot(x - xp, y - yp)[keep]),
    ))

tab = pd.DataFrame(filas).set_index('campaña')
display(tab.drop(columns=['p1', 'p2', 'p3', 'a0', 'E']).round(2))

,excluida,N,N_ok,pct_out,paridad,escala,x0,y0,incl,mio_med,mio_mae,cal_med,cal_mae,cal_med_px
campaña,,,,,,,,,,,,,,
10_2023-02,False,211,197,6.64,-1,6.50,787.08,533.27,1.62,5.69,6.84,13.69,26.34,2.00
10_2023-05,False,223,222,0.45,-1,6.51,787.26,534.57,4.61,5.06,5.31,9.64,29.05,1.41
12_2023-05,False,356,354,0.56,-1,6.41,800.99,479.20,1.51,12.68,13.54,14.04,20.29,2.24
13_2023-03,False,364,354,2.75,-1,6.44,795.00,443.56,1.69,6.00,6.42,8.77,18.63,1.41
14_2023-02,False,346,341,1.45,-1,6.50,783.81,498.32,2.18,5.90,6.70,12.81,21.45,2.00
15_2023-02,False,399,396,0.75,-1,6.48,768.44,492.65,2.36,5.14,5.80,9.46,13.44,1.41
16_2023-05,False,185,180,2.70,-1,6.46,791.54,502.62,3.23,5.09,5.93,13.49,31.59,2.00
17_2023-03,False,363,350,3.58,-1,6.51,805.07,488.57,1.41,6.32,7.09,9.10,22.08,1.41
19_2023-05,False,159,154,3.14,-1,6.47,820.87,477.64,2.21,6.48,8.16,12.99,49.33,2.24


## 3. Resumen global (mismos puntos, sin campañas excluidas)

In [ ]:
ok = (~df['outlier']) & (~df['excluida'])

def resumen(e):
    return pd.Series({'N': len(e), 'mediana': e.median(), 'MAE': e.mean(),
                      'RMSE': np.sqrt((e**2).mean()), 'P90': e.quantile(.9)})

display(pd.DataFrame({'outliers': [int((df['outlier'] & ~df['excluida']).sum())],
                      'de': [int((~df['excluida']).sum())],
                      'pct': [100*(df['outlier'] & ~df['excluida']).sum() / (~df['excluida']).sum()]}).round(2))
display(pd.DataFrame({'Mi ajuste': resumen(df.loc[ok, 'err_mio']),
                      'Caldas':    resumen(df.loc[ok, 'err_caldas'])}).T.round(2))

,outliers,de,pct
0,215,8773,2.45


,N,mediana,MAE,RMSE,P90
Mi ajuste,8558.0,7.17,9.16,12.00,17.53
Caldas,8558.0,14.10,29.18,55.24,71.79


## 4. Error por ángulo cenital

In [ ]:
df['banda_z'] = pd.cut(df['zenith'], [0, 30, 50, 60, 70, 80, 90])
por_z = df[ok].groupby('banda_z', observed=True).agg(
    N=('err_mio', 'size'),
    mio_med=('err_mio', 'median'), cal_med=('err_caldas', 'median'),
    mio_mae=('err_mio', 'mean'),   cal_mae=('err_caldas', 'mean'))
display(por_z.round(2))

,N,mio_med,cal_med,mio_mae,cal_mae
banda_z,,,,,
"(0, 30]",2425,7.22,14.17,8.79,28.85
"(30, 50]",3381,6.61,12.24,8.37,26.35
"(50, 60]",1603,7.18,15.37,9.19,28.19
"(60, 70]",919,8.99,17.72,11.05,33.70
"(70, 80]",223,14.16,43.12,16.45,61.93
"(80, 90]",7,30.97,83.73,31.37,99.46


## 5. ¿Qué es `Xpred`? Diagnóstico

Se ajusta el mismo modelo, pero a los pares (Xpred, Ypred) ↔ catálogo (sin outliers ni campañas excluidas).

- Si `Xpred` sale de una calibración geométrica suave, el residuo tiene que ser casi solo el redondeo a píxel entero (~0.3 px ≈ 2 arcmin).
- Si el residuo es grande, `Xpred` no es una proyección suave (o no es la de esa campaña).

También compara el centro de Caldas con el mío.

In [ ]:
filas_pred = []
for camp, g in df[ok].groupby('campaña'):
    xp, yp = g['Posicion_Xpred'].values.astype(float), g['Posicion_Ypred'].values.astype(float)
    fp = ajustar(xp, yp, g['zenith'].values, g['azimuth'].values)
    mio = params[camp]['p']
    filas_pred.append(dict(
        campaña=camp, N=len(g),
        resid_Xpred_med=np.median(fp['err']), resid_Xpred_p90=np.percentile(fp['err'], 90),
        x0_caldas=fp['p'][0], y0_caldas=fp['p'][1], x0_mio=mio[0], y0_mio=mio[1],
        dcentro_px=np.hypot(fp['p'][0] - mio[0], fp['p'][1] - mio[1]),
        incl_caldas=abs(fp['p'][6]), incl_mio=abs(mio[6]),
    ))
diag_pred = pd.DataFrame(filas_pred).set_index('campaña')
display(diag_pred.round(2))

,N,resid_Xpred_med,resid_Xpred_p90,x0_caldas,y0_caldas,x0_mio,y0_mio,dcentro_px,incl_caldas,incl_mio
campaña,,,,,,,,,,
10_2023-02,197,8.27,44.38,786.72,528.09,787.08,533.27,5.19,0.91,1.62
10_2023-05,222,10.33,65.43,783.71,531.69,787.26,534.57,4.58,4.58,4.61
12_2023-05,354,7.71,15.37,798.58,481.08,800.99,479.20,3.06,1.20,1.51
13_2023-03,354,4.46,14.18,796.64,440.17,795.00,443.56,3.77,1.34,1.69
14_2023-02,341,5.60,33.93,784.92,498.06,783.81,498.32,1.14,2.24,2.18
15_2023-02,396,4.73,12.08,767.49,493.79,768.44,492.65,1.47,2.52,2.36
16_2023-05,180,8.30,91.09,792.17,507.99,791.54,502.62,5.41,2.55,3.23
17_2023-03,350,5.80,30.75,804.73,488.41,805.07,488.57,0.37,1.44,1.41
19_2023-05,154,6.89,80.16,822.84,480.79,820.87,477.64,3.72,2.26,2.21


## 6. Guardar

In [ ]:
df.to_csv(os.path.join(PATH_BASE, "df_global_con_ajuste.csv"), index=False)
tab.to_csv(os.path.join(PATH_BASE, "ajuste_por_campaña.csv"))
display(pd.DataFrame({'archivo': ['df_global_con_ajuste.csv', 'ajuste_por_campaña.csv'],
                      'filas': [len(df), len(tab)]}))

,archivo,filas
0,df_global_con_ajuste.csv,8870
1,ajuste_por_campaña.csv,34
